<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Standard Supervised Fine-Tuning (SFT)

In [ ]:
# Mathematical Foundation: Cross-entropy loss for next token prediction
# Formula Loss: L_SFT = -∑ log π_θ(y_t|x, y_{<t})
# Method: Full parameter fine-tuning with supervised learning on task-specific data

"""
THEORETICAL FOUNDATION

Supervised Fine-Tuning (SFT) is the foundational method for adapting pre-trained language models
to specific tasks. It works by:

1. MATHEMATICAL BASIS:
   - Cross-entropy loss on next token prediction
   - Formula: L_SFT = -∑_{t=1}^T log P(y_t | x, y_1, ..., y_{t-1}; θ)
   - Optimization: θ* = argmin_θ E[(x,y)~D][L_SFT(x, y; θ)]

2. PROCESS:
   - Takes pre-trained model with general language understanding
   - Fine-tunes ALL parameters on curated instruction-following examples
   - Learns to map inputs to desired outputs through demonstration

3. ADVANTAGES:
   - Simple and interpretable training objective
   - Strong performance on target domain
   - Foundation for more advanced methods
   - Full model expressiveness maintained

4. LIMITATIONS:
   - Requires high-quality training data
   - May not align with human preferences without explicit examples
   - Computationally expensive (all parameters updated)
   - Risk of overfitting on small datasets

This implementation demonstrates SFT on cooking instruction data, showing how models learn
to provide structured, helpful responses through supervised learning.
"""

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DPOTrainer, SFTConfig, DPOConfig
from datasets import Dataset
import json
from datetime import datetime

# Global Parameters - Carefully tuned for 8GB VRAM
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
TEMPERATURE = 0.1  # Low for deterministic, high-quality responses
MAX_LENGTH = 1024  # Input sequence length for longer contexts
MAX_NEW_TOKENS = 1024  # For complete output generation
LEARNING_RATE_SFT = 1e-5  # Conservative learning rate for full fine-tuning
NUM_TRAIN_EPOCHS = 50  # Sufficient for convergence on small dataset
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # Memory constraint optimization
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size of 4
WARMUP_RATIO = 0.1  # Gradual learning rate warmup
LOGGING_STEPS = 10  # Regular progress monitoring
device = "cuda" if torch.cuda.is_available() else "cpu"

# Standard evaluation questions used across all notebooks
STANDARD_TEST_QUESTIONS = [
    "How do I cook perfect pasta?",
    "What's the secret to fluffy pancakes?",
    "How can I make my cookies soft and chewy?",
    "My bread never rises properly. Help!",
    "How do I prevent my cakes from being dry?",
]


def install_packages():
    """Install required packages for LLM fine-tuning"""
    packages = [
        "torch",
        "transformers>=4.35.0",
        "trl>=0.7.0",
        "peft>=0.6.0",
        "datasets",
        "bitsandbytes",
        "accelerate",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        except:
            pass


def cuda_usage():
    """Monitor CUDA memory usage for 8GB VRAM management"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        print(f"Available: {8.0 - reserved:.2f}GB remaining (assuming 8GB total)")
    else:
        print("CUDA not available - using CPU")


def cleanup_memory():
    """Comprehensive memory cleanup for CUDA memory management"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def simple_chat_test(
    model,
    tokenizer,
    prompt,
    temperature=TEMPERATURE,
    max_length=MAX_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """
    Test model with single prompt and return complete response

    Args:
        model: The language model to test
        tokenizer: Associated tokenizer
        prompt: Input text prompt
        temperature: Sampling temperature (lower = more deterministic)
        max_length: Maximum input length
        max_new_tokens: Maximum tokens to generate

    Returns:
        Complete model response as string
    """
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        with (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if torch.cuda.is_available()
            else torch.no_grad()
        ):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def test_model_comprehensive(model, tokenizer, model_name):
    """
    Comprehensive model evaluation with consistent formatting

    Returns:
        Dictionary with question-answer pairs for comparison
    """
    qa_results = {}

    print(f"\n" + "=" * 80)
    print(f"MODEL EVALUATION: {model_name}")
    print(f"=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\nQuestion {i}/{len(STANDARD_TEST_QUESTIONS)}: {question}")
        print("-" * 60)

        response = simple_chat_test(model, tokenizer, question)
        qa_results[question] = response

        print(f"Response:\n{response}")
        print("-" * 60)

    return qa_results


def compare_model_performance(base_results, trained_results, method_name):
    """
    Side-by-side comparison of base vs trained model outputs

    Args:
        base_results: QA results from base model
        trained_results: QA results from trained model
        method_name: Name of training method for headers
    """
    print(f"\n" + "=" * 80)
    print(f"COMPARATIVE ANALYSIS: Base Model vs {method_name}")
    print(f"=" * 80)
    print("This comparison shows how fine-tuning changes model behavior")
    print("Look for improvements in structure, detail, and helpfulness")
    print("=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 80)

        print(f"\n[BASE MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{base_results[question]}")

        print(f"\n[{method_name.upper()} MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{trained_results[question]}")

        print("\n" + "=" * 80)


def create_comprehensive_cooking_dataset():
    """
    Create high-quality cooking instruction dataset for SFT training

    Dataset Design Principles:
    - Clear question-answer format
    - Detailed, actionable instructions
    - Scientific explanations where relevant
    - Progressive complexity from basic to advanced
    - Consistent helpful tone and structure

    Returns:
        List of training examples with 'text' field
    """
    return [
        # Core Cooking Fundamentals
        {
            "text": "How do I cook perfect pasta?\n\nHere's the foolproof method:\n\n1. Use plenty of water (4-6 quarts per pound)\n2. Salt the water generously (it should taste like seawater)\n3. Bring to rolling boil before adding pasta\n4. Stir immediately and occasionally\n5. Test for doneness 1-2 minutes before package time\n6. Reserve pasta water before draining\n7. Never rinse unless making cold salad\n\nThe starchy pasta water helps sauce adhere beautifully!"
        },
        {
            "text": "What's the secret to fluffy pancakes?\n\nThe secret is gentle handling:\n\n1. Don't overmix - lumpy batter is perfect\n2. Let batter rest 5-10 minutes for fluffier texture\n3. Use room temperature ingredients for even mixing\n4. Add buttermilk or yogurt for tang and tenderness\n5. Cook on medium-low heat (325°F griddle)\n6. Wait for bubbles on surface before flipping\n7. Only flip once for best texture\n\nOvermixing develops gluten, making pancakes tough and dense."
        },
        {
            "text": "How can I make my cookies soft and chewy?\n\nFor perfectly soft cookies:\n\n1. Use more brown sugar than white (brown sugar retains moisture)\n2. Add an extra egg yolk for richness\n3. Use melted butter, then let dough cool\n4. Chill dough for 30+ minutes before baking\n5. Slightly underbake (edges set, centers soft)\n6. Cool on baking sheet for 5 minutes\n7. Store with a slice of bread to maintain softness\n\nBrown sugar's molasses keeps cookies tender longer than white sugar alone."
        },
        {
            "text": "My bread never rises properly. Help!\n\nTroubleshoot your yeast and environment:\n\n1. Check yeast expiration date\n2. Proof yeast in warm water (100-110°F) with pinch of sugar\n3. If no foam in 5-10 minutes, yeast is dead\n4. Use warm (not hot) liquids - hot kills yeast\n5. Create warm rising environment (oven light on)\n6. Allow enough time - first rise takes 1-2 hours\n7. Dough should double in size\n\nCold kitchens slow rising dramatically. Patience and warmth are key!"
        },
        {
            "text": "How do I prevent my cakes from being dry?\n\nMoist cake secrets:\n\n1. Don't overbake - toothpick should have few moist crumbs\n2. Use room temperature ingredients for better incorporation\n3. Add yogurt, sour cream, or buttermilk for moisture\n4. Don't overmix once flour is added\n5. Wrap cooled layers in plastic wrap overnight\n6. Simple syrup brushed on layers adds moisture\n7. Store covered to prevent drying\n\nMoisture comes from fats, acids, and proper mixing technique."
        },
        # Advanced Techniques
        {
            "text": "What's the best way to season food?\n\nSeasoning is layered throughout cooking:\n\n1. Salt early to draw out flavors\n2. Taste as you cook and adjust gradually\n3. Use acid (lemon, vinegar) to brighten flavors\n4. Toast spices before grinding for deeper flavor\n5. Add delicate herbs at the end\n6. Salt enhances sweetness and reduces bitterness\n7. Let seasoned dishes rest before final tasting\n\nGood seasoning balances salt, acid, fat, and heat harmoniously."
        },
        {
            "text": "How do I cook vegetables without making them mushy?\n\nKeep vegetables vibrant and crisp:\n\n1. Cut vegetables uniformly for even cooking\n2. Don't overcrowd the pan\n3. Use high heat for quick cooking methods\n4. Blanch and shock in ice water to stop cooking\n5. Add salt at the right time (not too early for tender veggies)\n6. Taste test frequently - they cook fast\n7. Remove from heat while slightly firm\n\nOvercooking breaks down cell walls, creating mushy texture."
        },
        {
            "text": "My scrambled eggs always turn out rubbery.\n\nFor silky, creamy eggs:\n\n1. Use low to medium-low heat only\n2. Add eggs to cold pan with butter\n3. Stir constantly with rubber spatula\n4. Remove from heat while still slightly wet\n5. Add cream or butter at the end\n6. Season with salt after cooking\n7. Be patient - good eggs take time\n\nHigh heat denatures proteins too quickly, creating rubber texture."
        },
        {
            "text": "How can I make my soups more flavorful?\n\nBuild layers of flavor:\n\n1. Start with aromatic base (onions, celery, carrots)\n2. Brown meat or vegetables for deeper flavor\n3. Deglaze pan to capture fond (browned bits)\n4. Use homemade or quality store-bought stock\n5. Add acid near the end to brighten\n6. Finish with fresh herbs\n7. Adjust seasoning after simmering\n\nTime allows flavors to meld and concentrate naturally."
        },
        {
            "text": "What's the trick to perfect rice every time?\n\nFoolproof rice method:\n\n1. Rinse rice until water runs clear\n2. Use 2:1 ratio (water to rice) for long grain\n3. Bring to boil, then reduce to lowest simmer\n4. Cover tightly and don't peek for 18 minutes\n5. Remove from heat, let rest 10 minutes\n6. Fluff with fork, never stir while cooking\n7. Season after cooking if desired\n\nSteam finishing makes rice fluffy, not sticky or mushy."
        },
        {
            "text": "How do I know when meat is properly cooked?\n\nSafe and delicious meat cooking:\n\n1. Use instant-read thermometer for accuracy\n2. Chicken: 165°F internal temperature\n3. Pork: 145°F with 3-minute rest\n4. Beef steaks: 125°F rare, 135°F medium-rare\n5. Let meat rest 5-10 minutes after cooking\n6. Juices should run clear for poultry\n7. Touch test: firm but yielding for medium doneness\n\nResting allows juices to redistribute throughout the meat."
        },
        {
            "text": "Why do my baked goods never turn out like the recipe?\n\nBaking is science - precision matters:\n\n1. Weigh ingredients instead of using cups\n2. Use room temperature ingredients unless specified\n3. Preheat oven fully (15-20 minutes)\n4. Don't open oven door during first 75% of baking\n5. Use proper pan size and material\n6. Check oven temperature with thermometer\n7. Follow recipe exactly first time, then modify\n\nBaking chemistry requires precise ratios to work properly."
        },
        # Professional Techniques
        {
            "text": "How do I make restaurant-quality steaks at home?\n\nProfessional steak techniques:\n\n1. Choose thick cuts (1-1.5 inches minimum)\n2. Bring to room temperature (30-45 minutes)\n3. Season generously with salt 40+ minutes before cooking\n4. Use cast iron or carbon steel pan\n5. Heat pan until smoking before adding oil\n6. Sear 2-3 minutes per side for crust\n7. Finish in 400°F oven if thick\n8. Rest 5-10 minutes before slicing\n\nThe Maillard reaction creates the flavorful crust that makes steaks special."
        },
        {
            "text": "What's the secret to crispy fried foods?\n\nAchieve perfect crispy texture:\n\n1. Use oil thermometer - maintain 350-375°F\n2. Don't overcrowd fryer - temperature drops\n3. Pat food completely dry before coating\n4. Double-coat for extra crunch (flour, egg, breadcrumbs)\n5. Let coated items rest 10 minutes before frying\n6. Fry in small batches for consistent results\n7. Drain on wire rack, not paper towels\n\nMoisture is the enemy of crispiness - control it at every step."
        },
        {
            "text": "How do I build rich, complex flavors in my cooking?\n\nLayering flavor techniques:\n\n1. Start with aromatic base (mirepoix: onions, celery, carrots)\n2. Brown proteins and vegetables for depth\n3. Use fond (browned bits) - deglaze and scrape\n4. Add herbs and spices at different stages\n5. Balance with acid, salt, fat, and heat\n6. Let flavors develop through slow cooking\n7. Taste and adjust throughout process\n\nBuilding flavors in layers creates depth that single additions can't match."
        },
    ]


def save_results_json(results, filename):
    """Save evaluation results to JSON for analysis"""
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": STANDARD_TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_training_summary():
    """Print theoretical summary of SFT method"""
    print("\n" + "=" * 80)
    print("STANDARD SUPERVISED FINE-TUNING (SFT) - THEORETICAL SUMMARY")
    print("=" * 80)
    print("Mathematical Foundation:")
    print("  Loss Function: L_SFT = -∑ log π_θ(y_t | x, y_<t)")
    print("  Optimization: Minimize cross-entropy on next token prediction")
    print("  Parameter Updates: All model weights (θ) updated via gradient descent")
    print()
    print("Key Characteristics:")
    print("  • Full parameter fine-tuning maintains maximum model expressiveness")
    print("  • Supervised learning on high-quality instruction-response pairs")
    print("  • Foundation method that other techniques build upon")
    print("  • Computationally intensive but provides strong baseline performance")
    print()
    print("Expected Outcomes:")
    print("  • Model learns to structure responses according to training examples")
    print("  • Improved coherence and relevance for target domain")
    print("  • Better instruction-following behavior")
    print("  • Potential for high-quality responses with sufficient training data")
    print("=" * 80)


def main():
    """
    Main training pipeline for Standard SFT

    Process:
    1. Load and test base model
    2. Prepare high-quality training dataset
    3. Configure training parameters for memory efficiency
    4. Train model using supervised fine-tuning
    5. Evaluate and compare results
    """
    print("=" * 80)
    print("STANDARD SUPERVISED FINE-TUNING (SFT)")
    print("=" * 80)
    print("Foundation method for LLM task adaptation")
    print("Mathematical basis: Cross-entropy loss on next token prediction")
    print("Approach: Full parameter fine-tuning on instruction-following examples")
    print("=" * 80)

    # Print theoretical foundation
    print_training_summary()

    install_packages()

    print(f"\nInitializing model: {MODEL_NAME}")
    print("Loading base model for evaluation and training...")

    # Load model with consistent dtype for stability
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )

    # Configure tokenizer padding
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<PAD>"})
        model.resize_token_embeddings(len(tokenizer))

    cuda_usage()

    # Save base model for future reference
    print("\nSaving base model for comparison...")
    os.makedirs("./models/base", exist_ok=True)
    model.save_pretrained("./models/base")
    tokenizer.save_pretrained("./models/base")

    # Evaluate base model performance
    print("\nEvaluating base model performance...")
    base_results = test_model_comprehensive(
        model, tokenizer, "Base Model (Pre-training Only)"
    )
    save_results_json(base_results, "base_model_results.json")

    # Prepare training dataset
    print(f"\nPreparing SFT training dataset...")
    sft_examples = create_comprehensive_cooking_dataset()
    sft_dataset = Dataset.from_list(sft_examples)

    print(f"Dataset Statistics:")
    print(f"  • Training examples: {len(sft_dataset)}")
    print(f"  • Domain: Cooking instructions and culinary techniques")
    print(f"  • Format: Question-answer pairs with detailed explanations")
    print(f"  • Quality: Manually curated for helpfulness and accuracy")

    # Configure SFT training
    training_args = SFTConfig(
        output_dir="./temp_sft",
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE_SFT,
        max_length=MAX_LENGTH,
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=False,
        bf16=torch.cuda.is_available(),
        dataloader_drop_last=True,
        warmup_ratio=WARMUP_RATIO,
        remove_unused_columns=False,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    print(f"\nStarting SFT training...")
    print("Training Details:")
    print(f"  • Epochs: {NUM_TRAIN_EPOCHS}")
    print(f"  • Learning Rate: {LEARNING_RATE_SFT}")
    print(
        f"  • Batch Size: {PER_DEVICE_TRAIN_BATCH_SIZE} (effective: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})"
    )
    print(f"  • Max Length: {MAX_LENGTH} tokens")
    print("\nLoss Function: Cross-entropy on next token prediction")
    print("Optimization: All model parameters updated via gradient descent")

    trainer.train()

    # Save trained model
    print(f"\nSaving SFT-trained model...")
    os.makedirs("./models/standard_sft", exist_ok=True)
    model.save_pretrained("./models/standard_sft")
    tokenizer.save_pretrained("./models/standard_sft")

    # Evaluate trained model
    print(f"\nEvaluating SFT-trained model...")
    trained_results = test_model_comprehensive(model, tokenizer, "SFT-Trained Model")
    save_results_json(trained_results, "sft_trained_results.json")

    # Comparative analysis
    compare_model_performance(base_results, trained_results, "Standard SFT")

    cleanup_memory()

    # Final summary
    print(f"\n" + "=" * 80)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print("Standard SFT Results:")
    print("  • Model learned to provide structured, detailed cooking advice")
    print("  • Responses show improved organization and helpfulness")
    print("  • Foundation established for advanced training methods")
    print()
    print("Files Created:")
    print("  • ./models/base/ - Original pre-trained model")
    print("  • ./models/standard_sft/ - SFT fine-tuned model")
    print("  • ./results/base_model_results.json - Base model evaluation")
    print("  • ./results/sft_trained_results.json - Trained model evaluation")
    print()
    print("Next Steps:")
    print("  • Compare with parameter-efficient methods (LoRA, DoRA)")
    print("  • Explore preference optimization (DPO, GRPO)")
    print("  • Analyze training efficiency and performance trade-offs")
    print("=" * 80)

In [2]:
# Run all
if __name__ == "__main__":
    main()

STANDARD SUPERVISED FINE-TUNING (SFT)
Foundation method for LLM task adaptation
Mathematical basis: Cross-entropy loss on next token prediction
Approach: Full parameter fine-tuning on instruction-following examples

STANDARD SUPERVISED FINE-TUNING (SFT) - THEORETICAL SUMMARY
Mathematical Foundation:
  Loss Function: L_SFT = -∑ log π_θ(y_t | x, y_<t)
  Optimization: Minimize cross-entropy on next token prediction
  Parameter Updates: All model weights (θ) updated via gradient descent

Key Characteristics:
  • Full parameter fine-tuning maintains maximum model expressiveness
  • Supervised learning on high-quality instruction-response pairs
  • Foundation method that other techniques build upon
  • Computationally intensive but provides strong baseline performance

Expected Outcomes:
  • Model learns to structure responses according to training examples
  • Improved coherence and relevance for target domain
  • Better instruction-following behavior
  • Potential for high-quality response

Truncating train dataset: 100%|██████████| 15/15 [00:00<00:00, 3623.07 examples/s]



Starting SFT training...
Training Details:
  • Epochs: 50
  • Learning Rate: 1e-05
  • Batch Size: 1 (effective: 4)
  • Max Length: 1024 tokens

Loss Function: Cross-entropy on next token prediction
Optimization: All model parameters updated via gradient descent


Step,Training Loss
10,2.808300
20,2.561100
30,1.866200
40,1.287100
50,0.802600
60,0.461400
70,0.224000
80,0.107100
90,0.062300
100,0.046900



Saving SFT-trained model...

Evaluating SFT-trained model...

MODEL EVALUATION: SFT-Trained Model

Question 1/5: How do I cook perfect pasta?
------------------------------------------------------------
Response:
Here's the foolproof method:

1. Use plenty of water (4-6 quarts per pound)
2. Salt the water generously (it should taste like seawater)
3. Bring to rolling boil before adding pasta
4. Stir immediately and occasionally
5. Test for doneness 1-2 minutes before package time
6. Reserve pasta water before draining
7. Never rinse unless making cold salad

The starchy pasta water helps sauce adhere beautifully!
------------------------------------------------------------

Question 2/5: What's the secret to fluffy pancakes?
------------------------------------------------------------
Response:
The perfect balance of fat, flour, and sweetness! Start by incorporating eggs for richness. Brown the flour in butter until golden brown. Add milk or buttermilk gradually while stirring. Add yo

___

References

In [3]:
# Citations
print(
    """
CITATIONS AND ACKNOWLEDGMENTS

Models and Frameworks:
• Qwen2 Model: Alibaba Cloud. "Qwen2 Technical Report." arXiv preprint arXiv:2407.10671 (2024).
• Transformers Library: Hugging Face. "Transformers: State-of-the-art Natural Language Processing." 
  Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations (2020).
• TRL (Transformer Reinforcement Learning): Hugging Face. "TRL: Transformer Reinforcement Learning Library."
  https://github.com/huggingface/trl

Core Technologies:
• PyTorch: Paszke, A., et al. "PyTorch: An imperative style, high-performance deep learning library." 
  Advances in Neural Information Processing Systems 32 (2019).
• BitsAndBytes: Dettmers, T., et al. "8-bit optimizers via block-wise quantization." 
  International Conference on Learning Representations (2022).
• Accelerate: Hugging Face. "Accelerate: A simple way to train and use PyTorch models with multi-GPU, TPU, mixed-precision."

Training Methodologies:
• Supervised Fine-tuning: Ouyang, L., et al. "Training language models to follow instructions with human feedback." 
  Advances in Neural Information Processing Systems 35 (2022).
• Parameter-Efficient Fine-tuning: Houlsby, N., et al. "Parameter-efficient transfer learning for NLP." 
  International Conference on Machine Learning (2019).

Hardware Optimization:
• 4-bit Quantization: Dettmers, T., et al. "QLoRA: Efficient Finetuning of Quantized LLMs." 
  arXiv preprint arXiv:2305.14314 (2023).
• Memory-Efficient Training: Rajbhandari, S., et al. "ZeRO: Memory optimizations toward training trillion parameter models." 
  International Conference for High Performance Computing, Networking, Storage and Analysis (2020).

Dataset and Evaluation:
• Instruction-following datasets inspired by techniques in: Wang, Y., et al. "Self-instruct: Aligning language model with self generated instructions." 
  Annual Meeting of the Association for Computational Linguistics (2022).

This implementation is for educational purposes and builds upon the research and development 
efforts of the machine learning community. All libraries and models are used according 
to their respective licenses.
"""
)


CITATIONS AND ACKNOWLEDGMENTS

Models and Frameworks:
• Qwen2 Model: Alibaba Cloud. "Qwen2 Technical Report." arXiv preprint arXiv:2407.10671 (2024).
• Transformers Library: Hugging Face. "Transformers: State-of-the-art Natural Language Processing." 
  Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations (2020).
• TRL (Transformer Reinforcement Learning): Hugging Face. "TRL: Transformer Reinforcement Learning Library."
  https://github.com/huggingface/trl

Core Technologies:
• PyTorch: Paszke, A., et al. "PyTorch: An imperative style, high-performance deep learning library." 
  Advances in Neural Information Processing Systems 32 (2019).
• BitsAndBytes: Dettmers, T., et al. "8-bit optimizers via block-wise quantization." 
  International Conference on Learning Representations (2022).
• Accelerate: Hugging Face. "Accelerate: A simple way to train and use PyTorch models with multi-GPU, TPU, mixed-precision."

Training Methodologie